# **Hackathon 1: Asistente inteligente con Llama**

## **Identificación morfológica de los principales mosquitos de importancia médica**

### **Objetivo**

Desarrollar un asistente basado en Llama que guíe la identificación morfológica de hembras adultas de mosquitos de importancia médica mediante preguntas sucesivas sobre caracteres diagnósticos, restringiendo la respuesta a cuatro taxones previamente definidos.

### **Especies consideradas**

- *Aedes aegypti*
- *Aedes albopictus*
- complejo *Anopheles gambiae*
- *Culex pipiens* sensu lato

El asistente utilizará información científica y reglas taxonómicas explícitas para controlar la identificación. Llama se empleará como componente conversacional del sistema y no sustituirá las reglas científicas de decisión.

## **Configuración del entorno**


### **Colab Secrets**


Para no exponer mi token directamente en el código, utilizo el panel de Secrets de Google Colab, identificado con el ícono de llave en la barra lateral izquierda. Ahí almaceno de forma segura mi token de Hugging Face, que utilizaré para acceder al modelo empleado en este proyecto.

In [ ]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl sentence-transformers --quiet
!pip uninstall -y torchao --quiet

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


## **Paso 1: Base de conocimiento (RAG)**

Indexo una base de conocimiento científica curada con sentence-transformers para recuperar evidencia asociada a los taxones considerados por el asistente. Cuando el motor taxonómico alcanza uno de los taxones objetivo, el RAG recupera el fragmento científico correspondiente junto con su fuente y página de origen. De esta manera, el resultado puede acompañarse de evidencia procedente de las publicaciones utilizadas.

In [ ]:
# Base de conocimiento científica curada para el RAG

documentos_rag = [
    {
        "id": "Coetzee_2020_gambiae",
        "fuente": "Coetzee_2020",
        "pagina": 10,
        "texto": (
            "3rd main dark area of wing vein 1 with a pale interruption, "
            "sometimes fused with preceding pale spot; scaling on abdomen "
            "very scanty, confined to tergum VIII or rarely VII. "
            "gambiae complex."
        ),
        "texto_busqueda": (
            "Complejo Anopheles gambiae. "
            "La tercera zona oscura principal de la vena 1 del ala presenta "
            "una interrupción clara. La escamación del abdomen es muy escasa "
            "y está confinada principalmente al terguito VIII o raramente VII."
        )
    },
    {
        "id": "Rueda_2004_aegypti",
        "fuente": "Rueda_2004",
        "pagina": 17,
        "texto": (
            "Scutum black or brown with a pair of submedian-longitudinal white stripes, "
            "but without median-longitudinal white stripe, or with white lyre-shaped markings; "
            "mesepimeron with two well separated white scale patches. "
            "Anterior portion of midfemur with a longitudinal white stripe. "
            "Clypeus with white scale patches. Aedes (Stegomyia) aegypti."
        ),
        "texto_busqueda": (
            "Aedes aegypti. "
            "El escudo presenta dos líneas blancas submedianas o un patrón blanco "
            "en forma de lira. El mesepimerón presenta dos parches de escamas blancas "
            "claramente separados. La parte anterior del fémur medio presenta una "
            "franja blanca longitudinal. El clípeo presenta escamas blancas."
        )
    },
    {
        "id": "Rueda_2004_albopictus",
        "fuente": "Rueda_2004",
        "pagina": 17,
        "texto": (
            "Scutum with a narrow median-longitudinal white stripe; "
            "mesepimeron with white scale patches not separated, forming a V-shaped white patch. "
            "Anterior portion of midfemur without a longitudinal white stripe. "
            "Clypeus without white scale patches. Aedes (Stegomyia) albopictus."
        ),
        "texto_busqueda": (
            "Aedes albopictus. "
            "El escudo presenta una sola línea blanca longitudinal media. "
            "Los parches blancos del mesepimerón no están separados y forman una V. "
            "La parte anterior del fémur medio no presenta una franja blanca longitudinal. "
            "El clípeo no presenta escamas blancas."
        )
    },
    {
        "id": "Ferreira_de_Freitas_2020_pipiens",
        "fuente": "Ferreira_de_Freitas_2020",
        "pagina": 6,
        "texto": (
            "Erect scales of dorsum of head pale medially, others dark; "
            "upper proepisternum with six to 12 setae; "
            "postpronotum usually with at least one seta pale or golden; "
            "mid lobe of scutellum with at least one seta pale or golden. "
            "Culex pipiens sensu lato."
        ),
        "texto_busqueda": (
            "Culex pipiens sensu lato. "
            "Las escamas erectas del dorso de la cabeza son pálidas medialmente. "
            "El proepisterno superior presenta entre seis y doce setas. "
            "El postpronoto y el lóbulo medio del escutelo presentan normalmente "
            "al menos una seta pálida o dorada."
        )
    }
]

print("Documentos RAG registrados:", len(documentos_rag))

Documentos RAG registrados: 4


In [ ]:
# Generar embeddings de la base de conocimiento científica curada

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

textos_rag = [
    documento["texto_busqueda"]
    for documento in documentos_rag
]

embeddings_rag = modelo_embeddings.encode(textos_rag)

print("Embeddings RAG generados:", embeddings_rag.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings RAG generados: (4, 384)


In [ ]:
# Buscar el documento científico más relacionado con una consulta

from sklearn.metrics.pairwise import cosine_similarity

def buscar_documento_rag(consulta):
    embedding_consulta = modelo_embeddings.encode([consulta])

    similitudes = cosine_similarity(
        embedding_consulta,
        embeddings_rag
    )[0]

    indice = np.argmax(similitudes)

    resultado = documentos_rag[indice].copy()
    resultado["similitud"] = float(similitudes[indice])

    return resultado

## **Paso 2: Motor taxonómico basado en reglas**

Utilizo un motor taxonómico basado en reglas explícitas para controlar la secuencia de identificación. La clave morfológica se incorpora directamente en el notebook como una estructura de Python, evitando dependencias de archivos externos. Cada respuesta determina la transición hacia el siguiente carácter morfológico o hacia un resultado terminal. De esta manera, la identificación no depende de que el modelo de lenguaje infiera el taxón, sino de una ruta de decisión previamente definida a partir de criterios científicos.

In [ ]:
# Definir la clave taxonómica estructurada dentro del notebook

clave_taxonomica = {
    "proyecto": {
        "nombre": "Asistente para identificación morfológica de mosquitos vectores de importancia médica",
        "alcance": "Hembras adultas",
        "taxones_objetivo": [
            "Aedes aegypti",
            "Aedes albopictus",
            "complejo Anopheles gambiae",
            "Culex pipiens sensu lato"
        ],
        "regla_general": (
            "No adivinar caracteres. Avanzar solo cuando el carácter sea observable. "
            "Si la información es insuficiente, contradictoria o el ejemplar queda "
            "fuera del alcance, detener la identificación."
        )
    },

    "fuentes": {
        "Huang_2001": {
            "cita_corta": "Huang, 2001",
            "titulo": (
                "A pictorial key for the identification of the subfamilies of Culicidae, "
                "genera of Culicinae, and subgenera of Aedes mosquitoes of the "
                "Afrotropical Region (Diptera: Culicidae)"
            ),
            "uso": "Separación taxonómica inicial y caracteres de género.",
            "limitacion": "Clave correspondiente a la región Afrotropical."
        },

        "Rueda_2004": {
            "cita_corta": "Rueda, 2004",
            "titulo": (
                "Pictorial keys for the identification of mosquitoes (Diptera: Culicidae) "
                "associated with Dengue Virus Transmission"
            ),
            "uso": "Separación de Aedes aegypti y Aedes albopictus en hembras adultas."
        },

        "Coetzee_2020": {
            "cita_corta": "Coetzee, 2020",
            "titulo": (
                "Key to the females of Afrotropical Anopheles mosquitoes "
                "(Diptera: Culicidae)"
            ),
            "uso": "Identificación hasta el complejo Anopheles gambiae.",
            "limitacion": (
                "La morfología de la hembra adulta no permite confirmar "
                "Anopheles gambiae sensu stricto."
            )
        },

        "Ferreira_de_Freitas_2020": {
            "cita_corta": "Ferreira-de-Freitas et al., 2020",
            "titulo": (
                "An Evaluation of Characters for the Separation of Two Culex Species "
                "(Diptera: Culicidae) Based on Material From the Upper Midwest"
            ),
            "uso": "Evaluación de caracteres compatibles con Culex pipiens sensu lato.",
            "limitacion": (
                "Criterios con alcance regional; conservar la formulación compatible "
                "con Culex pipiens sensu lato."
            )
        }
    },

    "clave_mosquitos": {
        "S0": {
            "pregunta": "¿Las antenas son densamente plumosas, con abundantes sedas largas?",
            "si": "FIN_MACHO",
            "no": "S1",
            "no_observable": "FIN_INSUFICIENTE"
        },

        "S1": {
            "pregunta": (
                "¿El margen posterior del scutellum es redondeado o ligeramente "
                "trilobulado, con las setas distribuidas de manera aproximadamente uniforme?"
            ),
            "si": "AN1",
            "no": "CU1",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Huang, 2001"
        },

        "AN1": {
            "pregunta": "¿Las patas presentan un patrón moteado, con zonas claras y oscuras?",
            "si": "AN2",
            "no": "FIN_OTRO_ANOPHELES",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Coetzee, 2020"
        },

        "AN2": {
            "pregunta": (
                "¿La tercera zona oscura principal de la vena 1 del ala presenta "
                "una interrupción clara?"
            ),
            "si": "AN3",
            "no": "FIN_OTRO_ANOPHELES",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Coetzee, 2020"
        },

        "AN3": {
            "pregunta": (
                "¿La escamación del abdomen es muy escasa y está principalmente limitada "
                "al terguito VIII, o raramente también al VII?"
            ),
            "si": "FIN_GAMBIAE",
            "no": "FIN_OTRO_ANOPHELES",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Coetzee, 2020"
        },

        "CU1": {
            "pregunta": "¿El pulvilo está bien desarrollado y tiene aspecto de pequeña almohadilla?",
            "si": "CU2",
            "no": "AE1",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Huang, 2001"
        },

        "CU2": {
            "pregunta": (
                "¿En el vértex hay numerosas escamas erectas bifurcadas que no están "
                "restringidas al occipucio?"
            ),
            "si": "CU3",
            "no": "AE1",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Huang, 2001"
        },

        "CU3": {
            "pregunta": (
                "¿El paratergito carece de escamas y el scutellum presenta escamas estrechas?"
            ),
            "si": "CU4",
            "no": "AE1",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Huang, 2001"
        },

        "CU4": {
            "pregunta": (
                "¿El primer flagelómero de la antena tiene aproximadamente la misma "
                "longitud que el segundo?"
            ),
            "si": "CU5",
            "no": "FIN_FUERA_ALCANCE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Huang, 2001"
        },

        "CU5": {
            "pregunta": (
                "¿Las escamas erectas del dorso de la cabeza presentan escamas pálidas "
                "en la zona media?"
            ),
            "si": "CU6",
            "no": "FIN_OTRO_CULEX",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Ferreira-de-Freitas et al., 2020"
        },

        "CU6": {
            "pregunta": (
                "¿El proepisterno superior presenta aproximadamente 6 a 12 setas y hay "
                "al menos una seta pálida o dorada en el postpronoto o en el lóbulo "
                "medio del scutellum?"
            ),
            "si": "FIN_PIPIENS",
            "no": "FIN_INSUFICIENTE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Ferreira-de-Freitas et al., 2020"
        },

        "AE1": {
            "pregunta": (
                "¿El scutum presenta dos líneas blancas submedianas o un patrón blanco "
                "semejante a una lira?"
            ),
            "si": "AE2",
            "no": "AE3",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Rueda, 2004"
        },

        "AE2": {
            "pregunta": (
                "¿Los dos parches blancos del mesepimeron están claramente separados?"
            ),
            "si": "AE4",
            "no": "FIN_INSUFICIENTE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Rueda, 2004"
        },

        "AE3": {
            "pregunta": (
                "¿El scutum presenta una sola línea blanca estrecha que recorre "
                "longitudinalmente la zona central?"
            ),
            "si": "AE5",
            "no": "FIN_FUERA_ALCANCE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Rueda, 2004"
        },

        "AE4": {
            "pregunta": (
                "¿La parte anterior del fémur medio presenta una franja blanca longitudinal "
                "y el clípeo tiene escamas blancas?"
            ),
            "si": "FIN_AEGYPTI",
            "no": "FIN_INSUFICIENTE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Rueda, 2004"
        },

        "AE5": {
            "pregunta": (
                "¿El fémur medio carece de franja blanca longitudinal, el clípeo carece "
                "de escamas blancas y los parches blancos del mesepimeron están unidos "
                "formando una V?"
            ),
            "si": "FIN_ALBOPICTUS",
            "no": "FIN_INSUFICIENTE",
            "no_observable": "FIN_INSUFICIENTE",
            "fuente": "Rueda, 2004"
        }
    },

    "resultados": {
        "FIN_MACHO": (
            "El ejemplar corresponde a un macho. Este asistente está diseñado "
            "para hembras adultas."
        ),
        "FIN_AEGYPTI": (
            "La combinación de caracteres es compatible con Aedes aegypti."
        ),
        "FIN_ALBOPICTUS": (
            "La combinación de caracteres es compatible con Aedes albopictus."
        ),
        "FIN_GAMBIAE": (
            "La combinación de caracteres es compatible con el complejo Anopheles gambiae. "
            "La morfología de la hembra adulta no permite confirmar "
            "Anopheles gambiae sensu stricto."
        ),
        "FIN_PIPIENS": (
            "La combinación de caracteres es compatible con Culex pipiens sensu lato."
        ),
        "FIN_OTRO_ANOPHELES": (
            "El ejemplar presenta caracteres de Anopheles, pero no coincide con el "
            "complejo Anopheles gambiae según los caracteres evaluados."
        ),
        "FIN_OTRO_CULEX": (
            "El ejemplar presenta caracteres de Culex, pero no permite asignarlo "
            "a Culex pipiens sensu lato."
        ),
        "FIN_FUERA_ALCANCE": (
            "Los caracteres observados no corresponden a ninguno de los cuatro "
            "taxones considerados por este asistente."
        ),
        "FIN_INSUFICIENTE": (
            "No es posible realizar una identificación confiable con los "
            "caracteres observados."
        )
    }
}

print(
    "Clave taxonómica integrada correctamente:",
    len(clave_taxonomica["clave_mosquitos"]),
    "nodos de decisión."
)

Clave taxonómica integrada correctamente: 16 nodos de decisión.


In [ ]:
# Avanzar un paso en la clave taxonómica

def avanzar_clave(nodo_actual, respuesta):
    respuesta = respuesta.strip().lower()

    if respuesta not in ["si", "no", "no_observable"]:
        return {
            "tipo": "error",
            "mensaje": "Respuesta no válida."
        }

    nodo = clave_taxonomica["clave_mosquitos"][nodo_actual]
    destino = nodo[respuesta]

    if destino.startswith("FIN_"):
        return {
            "tipo": "resultado",
            "estado": destino,
            "mensaje": clave_taxonomica["resultados"][destino]
        }

    siguiente_nodo = clave_taxonomica["clave_mosquitos"][destino]

    return {
        "tipo": "pregunta",
        "estado": destino,
        "pregunta": siguiente_nodo["pregunta"],
        "fuente": siguiente_nodo.get("fuente")
    }

In [ ]:
# Controlar el estado de una identificación

estado_identificacion = "S0"

def iniciar_identificacion():
    global estado_identificacion

    estado_identificacion = "S0"
    nodo = clave_taxonomica["clave_mosquitos"][estado_identificacion]

    return {
        "tipo": "pregunta",
        "estado": estado_identificacion,
        "pregunta": nodo["pregunta"],
        "fuente": nodo.get("fuente")
    }


def responder_identificacion(respuesta):
    global estado_identificacion

    if estado_identificacion is None:
        return {
            "tipo": "error",
            "mensaje": "La identificación ya terminó. Inicia una nueva identificación."
        }

    resultado = avanzar_clave(
        nodo_actual=estado_identificacion,
        respuesta=respuesta
    )

    if resultado["tipo"] == "pregunta":
        estado_identificacion = resultado["estado"]

    elif resultado["tipo"] == "resultado":
        estado_identificacion = None

    return resultado

## **Paso 3: Ajuste del modelo con LoRA**

En este proyecto ajusto un modelo ligero con LoRA para interpretar respuestas escritas en lenguaje natural y convertirlas en las categorías controladas que utiliza el motor taxonómico: si, no y no_observable. El desempeño se compara con el modelo base mediante ejemplos no utilizados para el entrenamiento. Durante el desarrollo se emplea un conjunto específico para analizar errores y evaluar el efecto del refuerzo, mientras que la evaluación final se realiza una sola vez con un conjunto reservado e independiente.

### **Evaluación del modelo base**

In [ ]:
# Cargar el modelo base sin ajustar

modelo_base_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base_id)

modelo_base = AutoModelForCausalLM.from_pretrained(
    modelo_base_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Modelo base cargado correctamente.")
print("Modelo:", modelo_base_id)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado correctamente.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
# Conjunto de evaluación reservado para comparar modelo base y modelo ajustado

evaluacion = [
    # si
    {
        "nodo": "S1",
        "respuesta": "Es ligeramente trilobulado.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE1",
        "respuesta": "Presenta un patrón blanco parecido a una lira.",
        "etiqueta": "si"
    },
    {
        "nodo": "AN1",
        "respuesta": "Las patas tienen zonas claras y oscuras.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU6",
        "respuesta": "Observo unas ocho setas y también una seta dorada.",
        "etiqueta": "si"
    },

    # no
    {
        "nodo": "S0",
        "respuesta": "Las antenas tienen pocas sedas y no son plumosas.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE3",
        "respuesta": "No hay una línea blanca longitudinal en el centro.",
        "etiqueta": "no"
    },
    {
        "nodo": "AN2",
        "respuesta": "La tercera zona oscura se observa continua.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU1",
        "respuesta": "El pulvilo es reducido y no tiene aspecto de almohadilla.",
        "etiqueta": "no"
    },

    # no_observable
    {
        "nodo": "AE2",
        "respuesta": "No alcanzo a distinguir si los parches están separados.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AN3",
        "respuesta": "El abdomen está dañado y no puedo evaluar las escamas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU5",
        "respuesta": "La cabeza no se aprecia con suficiente detalle.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE4",
        "respuesta": "No puedo observar bien ni el fémur medio ni el clípeo.",
        "etiqueta": "no_observable"
    }
]

print("Ejemplos de evaluación:", len(evaluacion))

Ejemplos de evaluación: 12


In [ ]:
# Clasificar una respuesta con el modelo base usando contexto taxonómico

def clasificar_respuesta_base(nodo, respuesta):
    pregunta = clave_taxonomica["clave_mosquitos"][nodo]["pregunta"]

    texto_contextual = (
        f"Pregunta taxonómica: {pregunta}\n"
        f"Respuesta del usuario: {respuesta}"
    )

    mensajes = [
        {
            "role": "system",
            "content": (
                "Clasifica la respuesta del usuario en una sola categoría: "
                "si, no o no_observable. "
                "Utiliza la pregunta taxonómica como contexto. "
                "Responde únicamente con una de esas tres etiquetas, sin explicación."
            )
        },
        {
            "role": "user",
            "content": texto_contextual
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(modelo_base.device)

    with torch.no_grad():
        salida = modelo_base.generate(
            **entradas,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    nuevos_tokens = salida[0][entradas["input_ids"].shape[1]:]

    respuesta_modelo = tokenizer.decode(
        nuevos_tokens,
        skip_special_tokens=True
    ).strip().lower()

    return respuesta_modelo

In [ ]:
# Evaluar el modelo base con el conjunto reservado contextual

predicciones_base = []
aciertos_base = 0

for ejemplo in evaluacion:
    prediccion = clasificar_respuesta_base(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    es_correcta = prediccion == ejemplo["etiqueta"]

    predicciones_base.append({
        "nodo": ejemplo["nodo"],
        "respuesta": ejemplo["respuesta"],
        "esperada": ejemplo["etiqueta"],
        "prediccion": prediccion,
        "correcta": es_correcta
    })

    if es_correcta:
        aciertos_base += 1

    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    print("Nodo:", ejemplo["nodo"])
    print("Pregunta:", pregunta)
    print("Respuesta:", ejemplo["respuesta"])
    print("Esperada:", ejemplo["etiqueta"])
    print("Predicción:", prediccion)
    print("Correcta:", es_correcta)
    print("-" * 60)

exactitud_base = aciertos_base / len(evaluacion)

print(f"\nAciertos del modelo base: {aciertos_base}/{len(evaluacion)}")
print(f"Exactitud del modelo base: {exactitud_base:.2%}")

Nodo: S1
Pregunta: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta: Es ligeramente trilobulado.
Esperada: si
Predicción: la respuesta del usuario es correcta.
Correcta: False
------------------------------------------------------------
Nodo: AE1
Pregunta: ¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?
Respuesta: Presenta un patrón blanco parecido a una lira.
Esperada: si
Predicción: la respuesta del usuario "el scut
Correcta: False
------------------------------------------------------------
Nodo: AN1
Pregunta: ¿Las patas presentan un patrón moteado, con zonas claras y oscuras?
Respuesta: Las patas tienen zonas claras y oscuras.
Esperada: si
Predicción: la respuesta del usuario es incorrecta.
Correcta: False
------------------------------------------------------------
Nodo: CU6
Pregunta: ¿El proepisterno superior presenta aproximadamente 6 a 

### **Preparación de los datos de entrenamiento**

In [ ]:
# Crear ejemplos de entrenamiento contextual para las tres categorías controladas

datos_entrenamiento = [
    # si
    {"nodo": "S1", "respuesta": "El margen se observa redondeado.", "etiqueta": "si"},
    {"nodo": "AN1", "respuesta": "Las patas muestran claramente zonas claras y oscuras.", "etiqueta": "si"},
    {"nodo": "AN2", "respuesta": "Se aprecia una interrupción en esa zona oscura de la vena.", "etiqueta": "si"},
    {"nodo": "CU1", "respuesta": "El pulvilo tiene aspecto de una pequeña almohadilla.", "etiqueta": "si"},
    {"nodo": "CU2", "respuesta": "Hay numerosas escamas erectas bifurcadas en el vértex.", "etiqueta": "si"},
    {"nodo": "CU4", "respuesta": "Los dos primeros flagelómeros tienen una longitud muy similar.", "etiqueta": "si"},
    {"nodo": "CU5", "respuesta": "Se observan escamas pálidas en la zona media de la cabeza.", "etiqueta": "si"},
    {"nodo": "AE1", "respuesta": "El scutum tiene un dibujo blanco parecido a una lira.", "etiqueta": "si"},
    {"nodo": "AE2", "respuesta": "Los dos parches blancos están claramente separados.", "etiqueta": "si"},
    {"nodo": "AE5", "respuesta": "No hay franja en el fémur ni escamas blancas en el clípeo y los parches forman una V.", "etiqueta": "si"},

    # no
    {"nodo": "S0", "respuesta": "Las antenas tienen pocas sedas y no presentan aspecto plumoso.", "etiqueta": "no"},
    {"nodo": "S1", "respuesta": "El margen posterior no tiene esa forma redondeada o trilobulada.", "etiqueta": "no"},
    {"nodo": "AN1", "respuesta": "Las patas presentan una coloración uniforme.", "etiqueta": "no"},
    {"nodo": "AN2", "respuesta": "La tercera zona oscura de la vena se observa continua.", "etiqueta": "no"},
    {"nodo": "AN3", "respuesta": "El abdomen presenta abundantes escamas en varios terguitos.", "etiqueta": "no"},
    {"nodo": "CU1", "respuesta": "El pulvilo es pequeño y poco desarrollado.", "etiqueta": "no"},
    {"nodo": "CU3", "respuesta": "El paratergito presenta escamas.", "etiqueta": "no"},
    {"nodo": "AE1", "respuesta": "No presenta líneas submedianas ni un patrón semejante a una lira.", "etiqueta": "no"},
    {"nodo": "AE3", "respuesta": "No se observa una línea blanca longitudinal en el centro del scutum.", "etiqueta": "no"},
    {"nodo": "AE4", "respuesta": "El fémur medio no tiene franja blanca y el clípeo carece de escamas blancas.", "etiqueta": "no"},

    # no_observable
    {"nodo": "S0", "respuesta": "Las antenas están dañadas y no puedo evaluar las sedas.", "etiqueta": "no_observable"},
    {"nodo": "S1", "respuesta": "No alcanzo a distinguir bien el margen posterior del scutellum.", "etiqueta": "no_observable"},
    {"nodo": "AN1", "respuesta": "Las patas no se observan completas en el ejemplar.", "etiqueta": "no_observable"},
    {"nodo": "AN2", "respuesta": "El ala está deteriorada y no puedo revisar esa vena.", "etiqueta": "no_observable"},
    {"nodo": "AN3", "respuesta": "No puedo evaluar la escamación porque el abdomen está dañado.", "etiqueta": "no_observable"},
    {"nodo": "CU2", "respuesta": "La cabeza no se aprecia con suficiente detalle.", "etiqueta": "no_observable"},
    {"nodo": "CU6", "respuesta": "No puedo contar las setas porque esa región no está visible.", "etiqueta": "no_observable"},
    {"nodo": "AE1", "respuesta": "El scutum está parcialmente oculto y no puedo reconocer el patrón.", "etiqueta": "no_observable"},
    {"nodo": "AE2", "respuesta": "No se distingue si los parches del mesepimeron están separados.", "etiqueta": "no_observable"},
    {"nodo": "AE5", "respuesta": "No puedo observar simultáneamente el fémur, el clípeo y el mesepimeron.", "etiqueta": "no_observable"}
]

print("Ejemplos de entrenamiento:", len(datos_entrenamiento))

Ejemplos de entrenamiento: 30


In [ ]:
# Preparar los datos contextuales en formato conversacional para el ajuste

from datasets import Dataset

def preparar_ejemplo(ejemplo):
    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    texto_contextual = (
        f"Pregunta taxonómica: {pregunta}\n"
        f"Respuesta del usuario: {ejemplo['respuesta']}"
    )

    mensajes = [
        {
            "role": "system",
            "content": (
                "Clasifica la respuesta del usuario en una sola categoría: "
                "si, no o no_observable. "
                "Utiliza la pregunta taxonómica como contexto. "
                "Responde únicamente con una de esas tres etiquetas, sin explicación."
            )
        },
        {
            "role": "user",
            "content": texto_contextual
        },
        {
            "role": "assistant",
            "content": ejemplo["etiqueta"]
        }
    ]

    texto_formateado = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": texto_formateado}


dataset_entrenamiento = Dataset.from_list(datos_entrenamiento)

dataset_entrenamiento = dataset_entrenamiento.map(
    preparar_ejemplo
)

print("Ejemplos preparados:", len(dataset_entrenamiento))
print("\nPrimer ejemplo:\n")
print(dataset_entrenamiento[0]["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Ejemplos preparados: 30

Primer ejemplo:

<|system|>
Clasifica la respuesta del usuario en una sola categoría: si, no o no_observable. Utiliza la pregunta taxonómica como contexto. Responde únicamente con una de esas tres etiquetas, sin explicación.</s>
<|user|>
Pregunta taxonómica: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta del usuario: El margen se observa redondeado.</s>
<|assistant|>
si</s>



### **Configuración de LoRA**

In [ ]:
# Configurar LoRA sobre el modelo base

from peft import LoraConfig, get_peft_model, TaskType

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]
)

modelo_ajustado = get_peft_model(
    modelo_base,
    config_lora
)

modelo_ajustado.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **Entrenamiento con LoRA**

In [ ]:
# Configurar el entrenamiento supervisado con LoRA

from trl import SFTConfig, SFTTrainer

config_entrenamiento = SFTConfig(
    output_dir="/content/tinyllama_lora_mosquitos",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
    dataset_text_field="text",
    max_length=256
)

trainer = SFTTrainer(
    model=modelo_ajustado,
    args=config_entrenamiento,
    train_dataset=dataset_entrenamiento,
    processing_class=tokenizer
)

print("Entrenador configurado correctamente.")

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Entrenador configurado correctamente.


In [ ]:
# Entrenar el modelo con LoRA

resultado_entrenamiento = trainer.train()

print("Entrenamiento finalizado.")

{'loss': '2.672', 'grad_norm': '2.032', 'learning_rate': '0.0002', 'entropy': '1.822', 'num_tokens': '584', 'mean_token_accuracy': '0.5362', 'epoch': '0.1333'}
{'loss': '2.741', 'grad_norm': '2.216', 'learning_rate': '0.000195', 'entropy': '1.811', 'num_tokens': '1134', 'mean_token_accuracy': '0.5312', 'epoch': '0.2667'}
{'loss': '2.7', 'grad_norm': '2.094', 'learning_rate': '0.00019', 'entropy': '1.955', 'num_tokens': '1729', 'mean_token_accuracy': '0.5368', 'epoch': '0.4'}
{'loss': '2.47', 'grad_norm': '2.072', 'learning_rate': '0.000185', 'entropy': '1.827', 'num_tokens': '2321', 'mean_token_accuracy': '0.561', 'epoch': '0.5333'}
{'loss': '2.619', 'grad_norm': '2.056', 'learning_rate': '0.00018', 'entropy': '1.903', 'num_tokens': '2936', 'mean_token_accuracy': '0.5444', 'epoch': '0.6667'}
{'loss': '2.335', 'grad_norm': '2.078', 'learning_rate': '0.000175', 'entropy': '1.803', 'num_tokens': '3566', 'mean_token_accuracy': '0.5722', 'epoch': '0.8'}
{'loss': '2.237', 'grad_norm': '2.406

In [ ]:
# Clasificar una respuesta con el modelo ajustado usando contexto taxonómico

def clasificar_respuesta_ajustada(nodo, respuesta):
    pregunta = clave_taxonomica["clave_mosquitos"][nodo]["pregunta"]

    texto_contextual = (
        f"Pregunta taxonómica: {pregunta}\n"
        f"Respuesta del usuario: {respuesta}"
    )

    mensajes = [
        {
            "role": "system",
            "content": (
                "Clasifica la respuesta del usuario en una sola categoría: "
                "si, no o no_observable. "
                "Utiliza la pregunta taxonómica como contexto. "
                "Responde únicamente con una de esas tres etiquetas, sin explicación."
            )
        },
        {
            "role": "user",
            "content": texto_contextual
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(modelo_ajustado.device)

    modelo_ajustado.eval()

    with torch.no_grad():
        salida = modelo_ajustado.generate(
            **entradas,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    nuevos_tokens = salida[0][entradas["input_ids"].shape[1]:]

    respuesta_modelo = tokenizer.decode(
        nuevos_tokens,
        skip_special_tokens=True
    ).strip().lower()

    return respuesta_modelo

In [ ]:
# Evaluar el modelo ajustado con el mismo conjunto reservado contextual

predicciones_ajustadas = []
aciertos_ajustados = 0

for ejemplo in evaluacion:
    prediccion = clasificar_respuesta_ajustada(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    es_correcta = prediccion == ejemplo["etiqueta"]

    predicciones_ajustadas.append({
        "nodo": ejemplo["nodo"],
        "respuesta": ejemplo["respuesta"],
        "esperada": ejemplo["etiqueta"],
        "prediccion": prediccion,
        "correcta": es_correcta
    })

    if es_correcta:
        aciertos_ajustados += 1

    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    print("Nodo:", ejemplo["nodo"])
    print("Pregunta:", pregunta)
    print("Respuesta:", ejemplo["respuesta"])
    print("Esperada:", ejemplo["etiqueta"])
    print("Predicción:", prediccion)
    print("Correcta:", es_correcta)
    print("-" * 60)

exactitud_ajustada = aciertos_ajustados / len(evaluacion)

print(
    f"\nAciertos del modelo ajustado: "
    f"{aciertos_ajustados}/{len(evaluacion)}"
)
print(
    f"Exactitud del modelo ajustado: "
    f"{exactitud_ajustada:.2%}"
)
print(
    f"Exactitud del modelo base: "
    f"{exactitud_base:.2%}"
)

Nodo: S1
Pregunta: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta: Es ligeramente trilobulado.
Esperada: si
Predicción: no_observable
Correcta: False
------------------------------------------------------------
Nodo: AE1
Pregunta: ¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?
Respuesta: Presenta un patrón blanco parecido a una lira.
Esperada: si
Predicción: si
Correcta: True
------------------------------------------------------------
Nodo: AN1
Pregunta: ¿Las patas presentan un patrón moteado, con zonas claras y oscuras?
Respuesta: Las patas tienen zonas claras y oscuras.
Esperada: si
Predicción: no_observable
Correcta: False
------------------------------------------------------------
Nodo: CU6
Pregunta: ¿El proepisterno superior presenta aproximadamente 6 a 12 setas y hay al menos una seta pálida o dorada en el postpronoto o en el lóbulo 

### **Evaluación de desarrollo**

In [ ]:
# Conjunto de desarrollo contextual para analizar errores del modelo ajustado

desarrollo = [
    # si
    {
        "nodo": "S1",
        "respuesta": "El borde se ve ligeramente trilobulado.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE1",
        "respuesta": "Se distinguen dos líneas blancas submedianas.",
        "etiqueta": "si"
    },
    {
        "nodo": "AN1",
        "respuesta": "Las patas muestran un patrón moteado.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU6",
        "respuesta": "Hay alrededor de diez setas y observo una seta clara.",
        "etiqueta": "si"
    },

    # no
    {
        "nodo": "S0",
        "respuesta": "Las antenas son simples y tienen pocas sedas.",
        "etiqueta": "no"
    },
    {
        "nodo": "AN2",
        "respuesta": "Esa zona oscura no presenta ninguna interrupción.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU2",
        "respuesta": "Las escamas erectas están solamente hacia el occipucio.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE3",
        "respuesta": "El scutum no tiene una línea blanca central.",
        "etiqueta": "no"
    },

    # no_observable
    {
        "nodo": "S1",
        "respuesta": "No puedo distinguir la forma del margen porque está dañado.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AN3",
        "respuesta": "El abdomen no está completo y no puedo evaluar las escamas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU5",
        "respuesta": "La cabeza está fuera de foco y no puedo ver el color de las escamas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE4",
        "respuesta": "No se observa con suficiente detalle ni el fémur medio ni el clípeo.",
        "etiqueta": "no_observable"
    }
]

print("Ejemplos de desarrollo:", len(desarrollo))

Ejemplos de desarrollo: 12


In [ ]:
# Evaluar el modelo ajustado con el conjunto de desarrollo contextual

aciertos_desarrollo = 0
resultados_desarrollo = []

for ejemplo in desarrollo:
    prediccion = clasificar_respuesta_ajustada(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    correcta = prediccion == ejemplo["etiqueta"]

    resultados_desarrollo.append({
        "nodo": ejemplo["nodo"],
        "respuesta": ejemplo["respuesta"],
        "esperada": ejemplo["etiqueta"],
        "prediccion": prediccion,
        "correcta": correcta
    })

    if correcta:
        aciertos_desarrollo += 1

    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    print("Nodo:", ejemplo["nodo"])
    print("Pregunta:", pregunta)
    print("Respuesta:", ejemplo["respuesta"])
    print("Esperada:", ejemplo["etiqueta"])
    print("Predicción:", prediccion)
    print("Correcta:", correcta)
    print("-" * 60)

exactitud_desarrollo = aciertos_desarrollo / len(desarrollo)

print(
    f"\nAciertos en desarrollo: "
    f"{aciertos_desarrollo}/{len(desarrollo)}"
)
print(
    f"Exactitud en desarrollo: "
    f"{exactitud_desarrollo:.2%}"
)

Nodo: S1
Pregunta: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta: El borde se ve ligeramente trilobulado.
Esperada: si
Predicción: no_observable
Correcta: False
------------------------------------------------------------
Nodo: AE1
Pregunta: ¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?
Respuesta: Se distinguen dos líneas blancas submedianas.
Esperada: si
Predicción: si
Correcta: True
------------------------------------------------------------
Nodo: AN1
Pregunta: ¿Las patas presentan un patrón moteado, con zonas claras y oscuras?
Respuesta: Las patas muestran un patrón moteado.
Esperada: si
Predicción: si
Correcta: True
------------------------------------------------------------
Nodo: CU6
Pregunta: ¿El proepisterno superior presenta aproximadamente 6 a 12 setas y hay al menos una seta pálida o dorada en el postpronoto o en el lóbulo medi

### **Refuerzo del entrenamiento**

A partir de los errores observados en el conjunto de desarrollo, se incorporan ejemplos adicionales para reforzar las tres categorías (`si`, `no` y `no_observable`). El adaptador LoRA continúa su entrenamiento con el conjunto ampliado y posteriormente se vuelve a evaluar sobre el mismo conjunto de desarrollo para comprobar si el refuerzo produce una mejora.

In [ ]:
# Ampliar los datos contextuales para reforzar las tres categorías

datos_refuerzo = [
    # si
    {
        "nodo": "S1",
        "respuesta": "La parte posterior se observa redondeada y con una ligera división en lóbulos.",
        "etiqueta": "si"
    },
    {
        "nodo": "AN2",
        "respuesta": "Se distingue un corte claro dentro de la tercera zona oscura.",
        "etiqueta": "si"
    },
    {
        "nodo": "AN3",
        "respuesta": "Casi no hay escamas en el abdomen y se concentran en el terguito VIII.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU1",
        "respuesta": "El pulvilo es evidente y tiene forma de pequeña almohadilla.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU2",
        "respuesta": "Hay muchas escamas erectas bifurcadas distribuidas por el vértex.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU5",
        "respuesta": "En la zona media de la cabeza aparecen varias escamas pálidas.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU6",
        "respuesta": "Cuento siete setas en el proepisterno y observo una seta dorada.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE2",
        "respuesta": "Los dos parches blancos no se tocan entre sí.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE3",
        "respuesta": "Hay una única franja blanca estrecha recorriendo el centro del scutum.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE4",
        "respuesta": "El fémur medio tiene una franja blanca y también veo escamas blancas en el clípeo.",
        "etiqueta": "si"
    },

    # no
    {
        "nodo": "S1",
        "respuesta": "El margen posterior es recto y no muestra lóbulos.",
        "etiqueta": "no"
    },
    {
        "nodo": "AN2",
        "respuesta": "La tercera zona oscura permanece continua de un extremo al otro.",
        "etiqueta": "no"
    },
    {
        "nodo": "AN3",
        "respuesta": "Veo abundantes escamas distribuidas en varios terguitos abdominales.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU1",
        "respuesta": "El pulvilo es rudimentario y no tiene aspecto de almohadilla.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU2",
        "respuesta": "Hay pocas escamas erectas y se concentran cerca del occipucio.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU5",
        "respuesta": "Las escamas erectas de la región media son todas oscuras.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU6",
        "respuesta": "Solo cuento tres setas y no observo ninguna seta pálida o dorada.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE2",
        "respuesta": "Los dos parches blancos están unidos entre sí.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE3",
        "respuesta": "El scutum presenta dos líneas blancas, no una sola línea central.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE4",
        "respuesta": "El fémur medio carece de franja blanca y el clípeo tampoco tiene escamas blancas.",
        "etiqueta": "no"
    },

    # no_observable
    {
        "nodo": "S1",
        "respuesta": "Falta la parte posterior del scutellum y no puedo determinar su forma.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AN2",
        "respuesta": "El ala está plegada justo sobre esa zona de la vena.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AN3",
        "respuesta": "Faltan varios segmentos abdominales y no puedo valorar la escamación.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU1",
        "respuesta": "El extremo de la pata no está conservado y el pulvilo no puede examinarse.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU2",
        "respuesta": "El vértex está fuera de foco y no puedo distinguir las escamas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU5",
        "respuesta": "La iluminación no permite distinguir si las escamas centrales son claras u oscuras.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU6",
        "respuesta": "La región del proepisterno está oculta y no puedo contar las setas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE2",
        "respuesta": "El mesepimeron está parcialmente cubierto y no puedo saber si los parches se separan.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE3",
        "respuesta": "Se han perdido muchas escamas del scutum y no puedo reconocer el patrón central.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE4",
        "respuesta": "El fémur medio y el clípeo quedan fuera de la imagen.",
        "etiqueta": "no_observable"
    }
]

print("Ejemplos de refuerzo:", len(datos_refuerzo))

Ejemplos de refuerzo: 30


In [ ]:
# Crear el conjunto de entrenamiento reforzado

datos_entrenamiento_reforzado = datos_entrenamiento + datos_refuerzo

print("Ejemplos originales:", len(datos_entrenamiento))
print("Ejemplos de refuerzo:", len(datos_refuerzo))
print("Total para el segundo entrenamiento:", len(datos_entrenamiento_reforzado))

Ejemplos originales: 30
Ejemplos de refuerzo: 30
Total para el segundo entrenamiento: 60


In [ ]:
# Preparar el conjunto de entrenamiento reforzado en formato conversacional

dataset_entrenamiento_reforzado = Dataset.from_list(
    datos_entrenamiento_reforzado
)

dataset_entrenamiento_reforzado = dataset_entrenamiento_reforzado.map(
    preparar_ejemplo
)

print(
    "Ejemplos preparados para el segundo entrenamiento:",
    len(dataset_entrenamiento_reforzado)
)

print("\nPrimer ejemplo:\n")
print(dataset_entrenamiento_reforzado[0]["text"])

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Ejemplos preparados para el segundo entrenamiento: 60

Primer ejemplo:

<|system|>
Clasifica la respuesta del usuario en una sola categoría: si, no o no_observable. Utiliza la pregunta taxonómica como contexto. Responde únicamente con una de esas tres etiquetas, sin explicación.</s>
<|user|>
Pregunta taxonómica: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta del usuario: El margen se observa redondeado.</s>
<|assistant|>
si</s>



In [ ]:
# Configurar el segundo entrenamiento con el conjunto reforzado

config_entrenamiento_reforzado = SFTConfig(
    output_dir="/content/tinyllama_lora_mosquitos_reforzado",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
    dataset_text_field="text",
    max_length=256
)

trainer_reforzado = SFTTrainer(
    model=modelo_ajustado,
    args=config_entrenamiento_reforzado,
    train_dataset=dataset_entrenamiento_reforzado,
    processing_class=tokenizer
)

print("Segundo entrenador configurado correctamente.")

Adding EOS to train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Segundo entrenador configurado correctamente.


In [ ]:
# Entrenar nuevamente el adaptador LoRA con el conjunto reforzado

resultado_entrenamiento_reforzado = trainer_reforzado.train()

print("Segundo entrenamiento finalizado.")

{'loss': '1.015', 'grad_norm': '2.12', 'learning_rate': '0.0001', 'entropy': '1.156', 'num_tokens': '614', 'mean_token_accuracy': '0.7952', 'epoch': '0.06667'}
{'loss': '0.8266', 'grad_norm': '1.954', 'learning_rate': '9.778e-05', 'entropy': '1.104', 'num_tokens': '1207', 'mean_token_accuracy': '0.8215', 'epoch': '0.1333'}
{'loss': '0.8751', 'grad_norm': '2.003', 'learning_rate': '9.556e-05', 'entropy': '1.092', 'num_tokens': '1792', 'mean_token_accuracy': '0.8236', 'epoch': '0.2'}
{'loss': '0.8113', 'grad_norm': '1.574', 'learning_rate': '9.333e-05', 'entropy': '1.008', 'num_tokens': '2369', 'mean_token_accuracy': '0.843', 'epoch': '0.2667'}
{'loss': '0.8616', 'grad_norm': '1.626', 'learning_rate': '9.111e-05', 'entropy': '1.024', 'num_tokens': '2966', 'mean_token_accuracy': '0.8263', 'epoch': '0.3333'}
{'loss': '0.7634', 'grad_norm': '1.452', 'learning_rate': '8.889e-05', 'entropy': '0.9501', 'num_tokens': '3560', 'mean_token_accuracy': '0.8406', 'epoch': '0.4'}
{'loss': '0.7689', 'g

### **Evaluación posterior al refuerzo**

Se evalúa nuevamente el modelo ajustado sobre el mismo conjunto de desarrollo para comprobar si los ejemplos adicionales produjeron una mejora respecto al desempeño previo al refuerzo.

In [ ]:
# Evaluar el modelo reforzado con el conjunto de desarrollo contextual

aciertos_desarrollo_reforzado = 0
resultados_desarrollo_reforzado = []

for ejemplo in desarrollo:
    prediccion = clasificar_respuesta_ajustada(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    correcta = prediccion == ejemplo["etiqueta"]

    resultados_desarrollo_reforzado.append({
        "nodo": ejemplo["nodo"],
        "respuesta": ejemplo["respuesta"],
        "esperada": ejemplo["etiqueta"],
        "prediccion": prediccion,
        "correcta": correcta
    })

    if correcta:
        aciertos_desarrollo_reforzado += 1

    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    print("Nodo:", ejemplo["nodo"])
    print("Pregunta:", pregunta)
    print("Respuesta:", ejemplo["respuesta"])
    print("Esperada:", ejemplo["etiqueta"])
    print("Predicción:", prediccion)
    print("Correcta:", correcta)
    print("-" * 60)

exactitud_desarrollo_reforzado = (
    aciertos_desarrollo_reforzado / len(desarrollo)
)

print(
    f"\nAciertos después del refuerzo: "
    f"{aciertos_desarrollo_reforzado}/{len(desarrollo)}"
)
print(
    f"Exactitud después del refuerzo: "
    f"{exactitud_desarrollo_reforzado:.2%}"
)
print(
    f"Exactitud antes del refuerzo: "
    f"{exactitud_desarrollo:.2%}"
)

Nodo: S1
Pregunta: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta: El borde se ve ligeramente trilobulado.
Esperada: si
Predicción: no
Correcta: False
------------------------------------------------------------
Nodo: AE1
Pregunta: ¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?
Respuesta: Se distinguen dos líneas blancas submedianas.
Esperada: si
Predicción: si
Correcta: True
------------------------------------------------------------
Nodo: AN1
Pregunta: ¿Las patas presentan un patrón moteado, con zonas claras y oscuras?
Respuesta: Las patas muestran un patrón moteado.
Esperada: si
Predicción: si
Correcta: True
------------------------------------------------------------
Nodo: CU6
Pregunta: ¿El proepisterno superior presenta aproximadamente 6 a 12 setas y hay al menos una seta pálida o dorada en el postpronoto o en el lóbulo medio del scute

### **Evaluación final**

La evaluación final utiliza un conjunto independiente de 12 respuestas que no intervino en el entrenamiento ni en el refuerzo. Sobre los mismos ejemplos se compara el modelo ajustado con LoRA frente al modelo base, desactivando temporalmente el adaptador para obtener la referencia sin ajuste.

In [ ]:
# Conjunto de evaluación final contextual e independiente

evaluacion_final = [
    # si
    {
        "nodo": "S1",
        "respuesta": "El margen posterior se aprecia ligeramente trilobulado.",
        "etiqueta": "si"
    },
    {
        "nodo": "AE1",
        "respuesta": "El patrón blanco del scutum tiene forma de lira.",
        "etiqueta": "si"
    },
    {
        "nodo": "AN1",
        "respuesta": "Se observan zonas claras y oscuras en las patas.",
        "etiqueta": "si"
    },
    {
        "nodo": "CU6",
        "respuesta": "Puedo contar unas nueve setas y una de ellas es pálida.",
        "etiqueta": "si"
    },

    # no
    {
        "nodo": "S0",
        "respuesta": "Las antenas son simples, con pocas sedas largas.",
        "etiqueta": "no"
    },
    {
        "nodo": "AN2",
        "respuesta": "La tercera zona oscura de la vena se ve continua.",
        "etiqueta": "no"
    },
    {
        "nodo": "CU1",
        "respuesta": "El pulvilo es pequeño y no parece una almohadilla.",
        "etiqueta": "no"
    },
    {
        "nodo": "AE3",
        "respuesta": "No presenta una línea blanca longitudinal en la zona central.",
        "etiqueta": "no"
    },

    # no_observable
    {
        "nodo": "S1",
        "respuesta": "Esa parte del scutellum está dañada y no puedo evaluar su forma.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AN3",
        "respuesta": "No puedo revisar la escamación porque falta parte del abdomen.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "CU5",
        "respuesta": "La región de la cabeza está borrosa y no permite distinguir el color de las escamas.",
        "etiqueta": "no_observable"
    },
    {
        "nodo": "AE4",
        "respuesta": "La imagen no permite observar con claridad el fémur medio ni el clípeo.",
        "etiqueta": "no_observable"
    }
]

print("Ejemplos de evaluación final:", len(evaluacion_final))

Ejemplos de evaluación final: 12


In [ ]:
# Clasificar una respuesta con LoRA desactivado usando contexto taxonómico

def clasificar_respuesta_sin_lora(nodo, respuesta):
    pregunta = clave_taxonomica["clave_mosquitos"][nodo]["pregunta"]

    texto_contextual = (
        f"Pregunta taxonómica: {pregunta}\n"
        f"Respuesta del usuario: {respuesta}"
    )

    mensajes = [
        {
            "role": "system",
            "content": (
                "Clasifica la respuesta del usuario en una sola categoría: "
                "si, no o no_observable. "
                "Utiliza la pregunta taxonómica como contexto. "
                "Responde únicamente con una de esas tres etiquetas, sin explicación."
            )
        },
        {
            "role": "user",
            "content": texto_contextual
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(modelo_ajustado.device)

    modelo_ajustado.eval()

    with modelo_ajustado.disable_adapter():
        with torch.no_grad():
            salida = modelo_ajustado.generate(
                **entradas,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

    nuevos_tokens = salida[0][entradas["input_ids"].shape[1]:]

    respuesta_modelo = tokenizer.decode(
        nuevos_tokens,
        skip_special_tokens=True
    ).strip().lower()

    return respuesta_modelo

In [ ]:
# Evaluar el modelo base y el modelo ajustado con el conjunto final contextual

aciertos_final_base = 0
aciertos_final_lora = 0

resultados_finales = []

for ejemplo in evaluacion_final:
    prediccion_base = clasificar_respuesta_sin_lora(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    prediccion_lora = clasificar_respuesta_ajustada(
        ejemplo["nodo"],
        ejemplo["respuesta"]
    )

    correcta_base = prediccion_base == ejemplo["etiqueta"]
    correcta_lora = prediccion_lora == ejemplo["etiqueta"]

    if correcta_base:
        aciertos_final_base += 1

    if correcta_lora:
        aciertos_final_lora += 1

    pregunta = clave_taxonomica["clave_mosquitos"][
        ejemplo["nodo"]
    ]["pregunta"]

    resultados_finales.append({
        "nodo": ejemplo["nodo"],
        "pregunta": pregunta,
        "respuesta": ejemplo["respuesta"],
        "esperada": ejemplo["etiqueta"],
        "base": prediccion_base,
        "base_correcta": correcta_base,
        "lora": prediccion_lora,
        "lora_correcta": correcta_lora
    })

    print("Nodo:", ejemplo["nodo"])
    print("Pregunta:", pregunta)
    print("Respuesta:", ejemplo["respuesta"])
    print("Esperada:", ejemplo["etiqueta"])
    print(
        "Modelo base:",
        prediccion_base,
        "| Correcta:",
        correcta_base
    )
    print(
        "Modelo LoRA:",
        prediccion_lora,
        "| Correcta:",
        correcta_lora
    )
    print("-" * 60)

exactitud_final_base = (
    aciertos_final_base / len(evaluacion_final)
)

exactitud_final_lora = (
    aciertos_final_lora / len(evaluacion_final)
)

print("\n--- Comparación final ---")

print(
    f"Modelo base: "
    f"{aciertos_final_base}/{len(evaluacion_final)} "
    f"= {exactitud_final_base:.2%}"
)

print(
    f"Modelo ajustado con LoRA: "
    f"{aciertos_final_lora}/{len(evaluacion_final)} "
    f"= {exactitud_final_lora:.2%}"
)

print(
    f"Mejora absoluta: "
    f"{(exactitud_final_lora - exactitud_final_base) * 100:.2f} "
    f"puntos porcentuales"
)

Nodo: S1
Pregunta: ¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?
Respuesta: El margen posterior se aprecia ligeramente trilobulado.
Esperada: si
Modelo base: la respuesta del usuario "el margen | Correcta: False
Modelo LoRA: si | Correcta: True
------------------------------------------------------------
Nodo: AE1
Pregunta: ¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?
Respuesta: El patrón blanco del scutum tiene forma de lira.
Esperada: si
Modelo base: la respuesta del usuario "el scut | Correcta: False
Modelo LoRA: si | Correcta: True
------------------------------------------------------------
Nodo: AN1
Pregunta: ¿Las patas presentan un patrón moteado, con zonas claras y oscuras?
Respuesta: Se observan zonas claras y oscuras en las patas.
Esperada: si
Modelo base: la respuesta del usuario "las pat | Correcta: False
Modelo LoRA: no | Correcta: Fal

### **Resultados del ajuste**

La evaluación final se realizó con un conjunto reservado de 12 respuestas que no se utilizó para el entrenamiento ni para el refuerzo posterior. La exactitud se calculó mediante coincidencia exacta entre la salida del modelo y la etiqueta esperada (si, no o no_observable). En esta ejecución, el modelo base obtuvo 0.00 %, mientras que el modelo ajustado con LoRA alcanzó 75.00 %, lo que representa una mejora absoluta de 75 puntos porcentuales.

Después del primer ajuste, LoRA alcanzó 50.00 % de exactitud. En el conjunto de desarrollo obtuvo 58.33 % antes del refuerzo y 66.67 % después de incorporar ejemplos adicionales. En la evaluación final independiente alcanzó 75.00 %. Aunque el desempeño mejoró respecto al modelo base, el modelo todavía cometió errores en 3 de las 12 respuestas finales; por ello, sus interpretaciones no se utilizan directamente para modificar la ruta taxonómica sin confirmación del usuario.

### **Decisiones metodológicas y limitaciones**

Durante el desarrollo se observó que la recuperación semántica sobre documentos completos podía devolver fragmentos ambiguos para una clave morfológica. Por ello, el RAG se restringió a fragmentos científicos previamente seleccionados de las fuentes utilizadas, manteniendo su referencia y página de origen.

El ajuste con LoRA se utilizó para interpretar respuestas del usuario en lenguaje natural. En la evaluación final independiente alcanzó 75.00 % de exactitud, pero todavía presentó errores en 3 de las 12 respuestas. Por ello, sus predicciones no modifican directamente una decisión taxonómica. La identificación permanece gobernada por reglas explícitas, mientras que las interpretaciones realizadas por LoRA requieren confirmación del usuario.

Esta arquitectura separa tres funciones: LoRA interpreta lenguaje natural, el motor de reglas controla la ruta de identificación y el RAG recupera evidencia científica asociada al resultado.

## **Paso 4: Integración del asistente**

Integro los componentes desarrollados previamente en un solo flujo. Las respuestas explícitas (si, no o no_observable) se envían directamente al motor taxonómico. Cuando el usuario responde en lenguaje libre, el modelo ajustado con LoRA propone una de esas categorías y solicita confirmación antes de avanzar por la clave estructurada. Cuando la identificación alcanza uno de los cuatro taxones objetivo, el RAG recupera el fragmento científico correspondiente para respaldar el resultado. De esta manera, LoRA apoya la interpretación del lenguaje natural, mientras que la decisión taxonómica permanece controlada por reglas explícitas.

In [ ]:
# Relacionar resultados taxonómicos con documentos del RAG

mapa_resultados_rag = {
    "FIN_AEGYPTI": "Rueda_2004_aegypti",
    "FIN_ALBOPICTUS": "Rueda_2004_albopictus",
    "FIN_GAMBIAE": "Coetzee_2020_gambiae",
    "FIN_PIPIENS": "Ferreira_de_Freitas_2020_pipiens"
}

print("Resultados vinculados al RAG:", len(mapa_resultados_rag))

Resultados vinculados al RAG: 4


In [ ]:
# Recuperar evidencia con RAG después de que el motor determina el taxón

def recuperar_evidencia_resultado(estado_resultado, mensaje_resultado):
    id_esperado = mapa_resultados_rag.get(estado_resultado)

    if id_esperado is None:
        return None

    # Recuperación semántica sobre la base científica curada
    evidencia = buscar_documento_rag(mensaje_resultado)

    # Verificación determinista: el RAG no puede cambiar el taxón decidido
    if evidencia["id"] != id_esperado:
        for documento in documentos_rag:
            if documento["id"] == id_esperado:
                evidencia = documento.copy()
                evidencia["similitud"] = None
                break

    return evidencia

In [ ]:
# Interpretar la respuesta sin permitir que LoRA avance automáticamente en la clave

import unicodedata


def interpretar_respuesta_usuario(texto):
    texto_normalizado = texto.strip().lower()

    texto_normalizado = "".join(
        caracter
        for caracter in unicodedata.normalize("NFD", texto_normalizado)
        if unicodedata.category(caracter) != "Mn"
    )

    # Respuestas afirmativas explícitas
    if (
        texto_normalizado == "si"
        or texto_normalizado.startswith("si,")
        or texto_normalizado.startswith("si.")
        or texto_normalizado.startswith("si;")
        or texto_normalizado.startswith("si:")
    ):
        return {
            "categoria": "si",
            "origen": "respuesta_explicita",
            "requiere_confirmacion": False
        }

    # Respuestas negativas explícitas
    # No usamos startswith("no ") porque frases como
    # "No puedo observar..." deben ser interpretadas por LoRA.
    if (
        texto_normalizado == "no"
        or texto_normalizado.startswith("no,")
        or texto_normalizado.startswith("no.")
        or texto_normalizado.startswith("no;")
        or texto_normalizado.startswith("no:")
    ):
        return {
            "categoria": "no",
            "origen": "respuesta_explicita",
            "requiere_confirmacion": False
        }

    # Imposibilidad de observación indicada explícitamente
    if texto_normalizado in [
        "no_observable",
        "no observable"
    ]:
        return {
            "categoria": "no_observable",
            "origen": "respuesta_explicita",
            "requiere_confirmacion": False
        }

    # Respuesta descriptiva: LoRA propone una categoría,
    # pero no modifica automáticamente la clave
    if (
        estado_identificacion is not None
        and estado_identificacion in clave_taxonomica["clave_mosquitos"]
    ):
        categoria = clasificar_respuesta_ajustada(
            estado_identificacion,
            texto
        ).strip().lower()

        if categoria not in ["si", "no", "no_observable"]:
            categoria = "no_observable"

        return {
            "categoria": categoria,
            "origen": "modelo_lora",
            "requiere_confirmacion": True
        }

    return {
        "categoria": None,
        "origen": "sin_identificacion_activa",
        "requiere_confirmacion": False
    }

### **Interacción con el asistente**

In [ ]:
# Ejecutar el asistente integrando LoRA, motor taxonómico y RAG

def ejecutar_asistente():
    inicio = iniciar_identificacion()

    print("\nAsistente de identificación morfológica")
    print("---------------------------------------")
    print(inicio["pregunta"])

    while estado_identificacion is not None:

        respuesta_usuario = input("\nTu respuesta: ").strip()

        if respuesta_usuario.lower() == "salir":
            print("Identificación finalizada por el usuario.")
            break

        interpretacion = interpretar_respuesta_usuario(respuesta_usuario)
        categoria = interpretacion["categoria"]

        if categoria is None:
            print("No hay una identificación activa.")
            continue

        # Si LoRA interpreta una respuesta libre,
        # el usuario debe confirmar antes de avanzar
        if interpretacion["requiere_confirmacion"]:

            print(
                f"\nInterpretación propuesta por LoRA: {categoria}"
            )

            confirmacion = input(
                "¿Confirmas esta interpretación? (si/no): "
            ).strip().lower()

            if confirmacion == "no":
                categoria = input(
                    "Indica la categoría correcta "
                    "(si/no/no_observable): "
                ).strip().lower()

                if categoria not in ["si", "no", "no_observable"]:
                    print("Categoría no válida. La clave no avanzó.")
                    continue

            elif confirmacion != "si":
                print("Confirmación no válida. La clave no avanzó.")
                continue

        resultado = responder_identificacion(categoria)

        print(f"Respuesta utilizada: {categoria}")

        if resultado["tipo"] == "pregunta":
            print("\n" + resultado["pregunta"])

        elif resultado["tipo"] == "resultado":
            print("\nResultado:")
            print(resultado["mensaje"])

            if resultado["estado"] in mapa_resultados_rag:

                evidencia = recuperar_evidencia_resultado(
                    resultado["estado"],
                    resultado["mensaje"]
                )

                if evidencia is not None:
                    print("\nEvidencia científica:")
                    print("Fuente:", evidencia["fuente"])
                    print("Página:", evidencia["pagina"])
                    print("Fragmento:", evidencia["texto"])

### **Prueba interactiva del asistente**

El asistente recorre la clave morfológica mediante preguntas sucesivas. Las respuestas explícitas (`si`, `no` o `no_observable`) se envían directamente al motor taxonómico. Cuando el usuario responde con lenguaje libre, el modelo ajustado con LoRA propone una interpretación que debe ser confirmada antes de modificar la ruta de identificación. Al alcanzar uno de los taxones objetivo, el sistema recupera mediante RAG el fragmento de evidencia científica correspondiente.

In [ ]:
# Iniciar el asistente interactivo

ejecutar_asistente()


Asistente de identificación morfológica
---------------------------------------
¿Las antenas son densamente plumosas, con abundantes sedas largas?

Tu respuesta: no
Respuesta utilizada: no

¿El margen posterior del scutellum es redondeado o ligeramente trilobulado, con las setas distribuidas de manera aproximadamente uniforme?

Tu respuesta: no
Respuesta utilizada: no

¿El pulvilo está bien desarrollado y tiene aspecto de pequeña almohadilla?

Tu respuesta: no
Respuesta utilizada: no

¿El scutum presenta dos líneas blancas submedianas o un patrón blanco semejante a una lira?

Tu respuesta: El scutum muestra un patrón blanco en forma de lira

Interpretación propuesta por LoRA: no
¿Confirmas esta interpretación? (si/no): no
Indica la categoría correcta (si/no/no_observable): si
Respuesta utilizada: si

¿Los dos parches blancos del mesepimeron están claramente separados?

Tu respuesta: si
Respuesta utilizada: si

¿La parte anterior del fémur medio presenta una franja blanca longitudinal 

## **Fuentes científicas**

- Huang, Y.-M. (2001). *A pictorial key for the identification of the subfamilies of Culicidae, genera of Culicinae, and subgenera of Aedes mosquitoes of the Afrotropical Region (Diptera: Culicidae).*
- Rueda, L. M. (2004). *Pictorial keys for the identification of mosquitoes (Diptera: Culicidae) associated with Dengue Virus Transmission.*
- Coetzee, M. (2020). *Key to the females of Afrotropical Anopheles mosquitoes (Diptera: Culicidae).*
- Ferreira-de-Freitas, L., Thrun, N. B., Tucker, B., Melidosian, L., & Bartholomay, L. C. (2020). *An Evaluation of Characters for the Separation of Two Culex Species (Diptera: Culicidae) Based on Material From the Upper Midwest.*